In [1]:
import json
import re

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
whole_dataset = pd.read_csv("~/Box/dsi-core/11th-hour/good-food-purchasing/nov2025-dataset/cleaned-by-vy.csv", dtype=str).fillna("")
whole_dataset["Product Type"] = whole_dataset["Product Type"].apply(lambda x: re.sub(r"\s+", " ", x.upper()))
whole_dataset["sub_types"] = [[x for x in [row["Sub-Type 1"], row["Sub-Type 2"], row["Sub-Type 3"]] if x != ""] for _, row in whole_dataset.iterrows()]

rng = np.random.default_rng(12345)
whole_dataset = whole_dataset.iloc[rng.permutation(np.arange(len(whole_dataset)))]

In [3]:
len(whole_dataset)

104060

In [4]:
len(whole_dataset) / 3

34686.666666666664

In [5]:
whole_dataset.columns

Index(['Product Type', 'Food Product Group', 'Food Product Category',
       'Primary Food Product Category', 'Product Name', 'Basic Type',
       'Sub-Type 1', 'Sub-Type 2', 'Sub-Type 3', 'Flavor/Cut', 'Shape', 'Skin',
       'Seed/Bone', 'Processing', 'Cooked/Cleaned', 'WG/WGR',
       'Dietary Concern', 'Additives', 'Dietary Accommodation', 'Frozen',
       'Packaging', 'Commodity', 'Multiple Names', 'sub_types'],
      dtype='object')

In [6]:
category_dataset = whole_dataset[["Product Type", "Food Product Category", "Primary Food Product Category"]].copy()
category_dataset["Primary Food Product Category"] = np.where(
    category_dataset["Food Product Category"] == category_dataset["Primary Food Product Category"],
    "",
    category_dataset["Primary Food Product Category"],
)
category_dataset = category_dataset.rename(columns={"Product Type": "input", "Food Product Category": "category", "Primary Food Product Category": "subcategory"})

In [7]:
category_dataset

,input,category,subcategory
14261,BGLSHP WHP CRM,Cheese,
27496,CHICKEN WHL GRN MNDRN ORNGE FC 15552-4,Chicken,
50493,"ICE CREAM MIX, SOFT SERVE CHOC 4%",Milk & Dairy,
89632,SPICE-ZAATAR SST .9 LB,Condiments & Snacks,
58543,"MEAL MART CHEESE RAVIOLI DINNER, GLATT KOSHER ...",Meals,Cheese
...,...,...,...
9346,BANANA READY TO GO NO 6 40 LB,Fruit,
61401,MUFFIN TOP APPLE CINNAMON W/G I/W 63111,Condiments & Snacks,
70421,PEPPER RED FIRE RSTD PIECES,Condiments & Snacks,
50398,"ICE CREAM CUP, BRWNE CHOC FDGE",Milk & Dairy,


In [8]:
BATCH_SIZES = [1000, 800, 600] + [300] * 2 + [200] * 3 + [100] * 6 + list(rng.integers(2, 10, len(whole_dataset)) * 10)

In [9]:
def message_pairs(df):
    out = []

    start_index = 0
    for batch_size in BATCH_SIZES:
        stop_index = start_index + batch_size
        if start_index >= len(df):
            break
        batch = df[start_index:stop_index]
        start_index = stop_index

        user_message = []
        assistant_message = []
        for _, row in batch.iterrows():
            user_message.append({"input": row["input"]})
            assistant_message.append({column: row[column] for column in batch.columns if row[column]})

        out.append([
            {"role": "user", "content": json.dumps({"food_products": user_message})},
            {"role": "assistant", "content": json.dumps({"food_products": assistant_message})},
        ])

    return out

In [10]:
with open("fine-tuning-2/category-training-try2.jsonl", "w") as file:
    for message_pair in message_pairs(category_dataset[:30000]):
        file.write(json.dumps({"messages": message_pair}) + "\n")

In [11]:
with open("fine-tuning-2/category-validation-try2.jsonl", "w") as file:
    for message_pair in message_pairs(category_dataset[30000:60000]):
        file.write(json.dumps({"messages": message_pair}) + "\n")

In [12]:
with open("fine-tuning-2/category-testing-try2.jsonl", "w") as file:
    for message_pair in message_pairs(category_dataset[60000:]):
        file.write(json.dumps({"messages": message_pair}) + "\n")

In [13]:
tag_dataset = whole_dataset[["Product Type", "Basic Type", "sub_types"]].rename(columns={"Product Type": "input", "Basic Type": "basic_type"})
tag_dataset["flavored"] = whole_dataset["Flavor/Cut"] == "flavored"
tag_dataset["shape"] = whole_dataset["Shape"]
tag_dataset["meat_cut"] = np.where(whole_dataset["Flavor/Cut"] == "flavored", "", whole_dataset["Flavor/Cut"])
tag_dataset["meat_skin"] = whole_dataset["Skin"]
tag_dataset["meat_bone"] = whole_dataset["Seed/Bone"] == "bone-in"
tag_dataset["seed_pitted"] = whole_dataset["Seed/Bone"] == "pitted"
tag_dataset["processing"] = whole_dataset["Processing"]
tag_dataset["cooked"] = whole_dataset["Cooked/Cleaned"]
tag_dataset["whole_grain"] = whole_dataset["WG/WGR"] == "whole grain rich"
tag_dataset["fat_content"] = np.where(whole_dataset["Dietary Concern"].isin(["nonfat", "low fat", "1%", "2%", "fat free"]), whole_dataset["Dietary Concern"], "")
tag_dataset["sodium_level"] = np.where(whole_dataset["Dietary Concern"].isin(["low sodium", "reduced sodium", "salted", "unsalted"]), whole_dataset["Dietary Concern"], "")
tag_dataset["caffeine"] = np.where(whole_dataset["Dietary Concern"].isin(["decaffeinated", "caffeinated"]), whole_dataset["Dietary Concern"], "")
tag_dataset["diet"] = np.where(whole_dataset["Dietary Concern"].isin(["diet", "reduced calorie"]), whole_dataset["Dietary Concern"], "")
tag_dataset["reduced_sugar"] = whole_dataset["Dietary Concern"] == "reduced sugar"
tag_dataset["sweetened"] = np.where(whole_dataset["Additives"].isin(["sweetened", "unsweetened"]), whole_dataset["Additives"], "")
tag_dataset["additives"] = np.where(whole_dataset["Additives"].isin(["additives", "no additives"]), whole_dataset["Additives"], "")
tag_dataset["dietary_accommodation"] = whole_dataset["Dietary Accommodation"]
tag_dataset["frozen"] = whole_dataset["Frozen"]
tag_dataset["packaging"] = whole_dataset["Packaging"]
tag_dataset["commodity"] = whole_dataset["Commodity"] == "commodity"

In [14]:
tag_dataset

,input,basic_type,sub_types,flavored,shape,meat_cut,meat_skin,meat_bone,seed_pitted,processing,...,sodium_level,caffeine,diet,reduced_sugar,sweetened,additives,dietary_accommodation,frozen,packaging,commodity
14261,BGLSHP WHP CRM,cream cheese,[],False,,,,False,False,whipped,...,,,,False,,,,,,False
27496,CHICKEN WHL GRN MNDRN ORNGE FC 15552-4,chicken,[mandarin orange],False,,,,False,False,breaded,...,,,,False,,,,,,False
50493,"ICE CREAM MIX, SOFT SERVE CHOC 4%",ice cream,[mix],False,,,,False,False,,...,,,,False,,,,,,False
89632,SPICE-ZAATAR SST .9 LB,seasoned,[zaatar],False,,,,False,False,,...,,,,False,,,,,,False
58543,"MEAL MART CHEESE RAVIOLI DINNER, GLATT KOSHER ...",entrÃ©e,[ravioli],False,,,,False,False,,...,,,,False,,,kosher,,,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9346,BANANA READY TO GO NO 6 40 LB,banana,[],False,,,,False,False,,...,,,,False,,,,,,False
61401,MUFFIN TOP APPLE CINNAMON W/G I/W 63111,muffin,[],False,,,,False,False,,...,,,,False,,,,,ss,False
70421,PEPPER RED FIRE RSTD PIECES,pepper,[],False,cut,,,False,False,,...,,,,False,,,,,,False
50398,"ICE CREAM CUP, BRWNE CHOC FDGE",ice cream,[chocolate fudge brownie],False,,,,False,False,,...,,,,False,,,,,ss,False


In [15]:
with open("fine-tuning-2/tag-training-try2.jsonl", "w") as file:
    for message_pair in message_pairs(tag_dataset[:30000]):
        file.write(json.dumps({"messages": message_pair}) + "\n")

In [16]:
with open("fine-tuning-2/tag-validation-try2.jsonl", "w") as file:
    for message_pair in message_pairs(tag_dataset[30000:60000]):
        file.write(json.dumps({"messages": message_pair}) + "\n")

In [17]:
with open("fine-tuning-2/tag-testing-try2.jsonl", "w") as file:
    for message_pair in message_pairs(tag_dataset[60000:]):
        file.write(json.dumps({"messages": message_pair}) + "\n")